# LoopMoE 1B (Test Branch) — Colab TPU v6e-1 MFU & HBM Benchmark

Measures steady-state **MFU%**, **tokens/sec**, **XLA compile time**, and **peak HBM** at uniform **4K (`R=1`)** with **Tokamax Megablox `ragged_dot`** + **Splash Attention** in native **BF16**.

In [ ]:
# 1. Install TPU v6e high-MFU JAX + OpenXLA Tokamax stack
!pip install -q -U "jax[tpu]" -f https://storage.googleapis.com/jax-releases/libtpu_releases.html
!pip install -q -U tokamax flax optax tokenizers datasets huggingface_hub pyarrow fsspec

In [ ]:
# 2. Clone the Test branch
import os
REPO_DIR = '/content/M3_Test'
if not os.path.exists(REPO_DIR):
    !git clone -b Test https://github.com/GarD0m/M3.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull origin Test

In [ ]:
# 3. Run full 1B MFU & HBM benchmark (4K R=1)
%cd {REPO_DIR}
!python scripts/bench_tpu_mfu.py \
    --tpu-preset v6e-1 \
    --seq-lens 4096 \
    --loops 1 \
    --microbatch 8 \
    --grad-accum 8 \
    --chunk-ce-size 16384 \
    --warmup-steps 2 \
    --bench-steps 5 \
    --out-json bench_tpu_v6e_results.json

In [ ]:
# 4. Optional: test grad_accum=16 with microbatch=8 at 4K R=1
%cd {REPO_DIR}
!python scripts/bench_tpu_mfu.py \
    --tpu-preset v6e-1 \
    --seq-lens 4096 \
    --loops 1 \
    --microbatch 8 \
    --grad-accum 16 \
    --chunk-ce-size 16384 \
    --warmup-steps 1 \
    --bench-steps 3